In [1]:
import os
import logging 
import requests 
import pymysql
import pandas as pd 
from dotenv import load_dotenv
from requests.exceptions import HTTPError, RequestException, Timeout

In [2]:
# Load the environment variables 
load_dotenv()

API_KEY         =   os.getenv("API_KEY")
API_HOST        =   os.getenv("API_HOST")

LEAGUE_CODE     =   os.getenv("LEAGUE_CODE", "PL")

MYSQL_DATABASE         =   os.getenv("MYSQL_DATABASE")
MYSQL_USERNAME         =   os.getenv("MYSQL_USERNAME")
MYSQL_PASSWORD         =   os.getenv("MYSQL_PASSWORD")
MYSQL_HOST             =   os.getenv("MYSQL_HOST")
MYSQL_PORT             =   os.getenv("MYSQL_PORT")

In [3]:
# Set up the logger 
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Create a file handler
file_handler = logging.FileHandler('football_table_standings.log')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Create a console handler
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Instantiate the logger object
logger = logging.getLogger()

# Add the file handler to the logger
logger.addHandler(file_handler)

# Add the console handler to the logger
logger.addHandler(console_handler)

In [4]:
url           =   f"https://api.football-data.org/v4/competitions/{LEAGUE_CODE}/standings"
headers       =   {"X-Auth-Token": API_KEY}
query_string  =   {}

In [5]:
try:
    api_response = requests.get(url, headers=headers, params=query_string, timeout=15)
    api_response.raise_for_status() 

except HTTPError as http_err:
    logger.error(f'HTTP error occurred: {http_err}')

except Timeout:
    logger.error('Request timed out after 15 seconds')

except RequestException as request_err:
    logger.error(f'Request error occurred: {request_err}')

In [6]:

standings_data          =   api_response.json()['standings']
#print(standings_data)

In [7]:

# standings is the outer list
table = standings_data[0]['table']

data_list = []

# Parsing through each team's standing
for team_info in table:
    rank            = team_info['position']
    team_name       = team_info['team']['name']
    played          = team_info['playedGames']
    win             = team_info['won']
    draw            = team_info['draw']
    lose            = team_info['lost']
    goals_for       = team_info['goalsFor']
    goals_against   = team_info['goalsAgainst']
    goals_diff      = team_info['goalDifference']
    points          = team_info['points']
    
    data_list.append([
        rank, team_name, played, win, draw, lose,
        goals_for, goals_against, goals_diff, points
    ])


In [8]:
# Create the dataFrame
columns         =   ['P', 'Team', 'GP', 'W', 'D', 'L', 'F', 'A', 'GD', 'Pts']
standings_df    =   pd.DataFrame(data_list, columns=columns)

# Display the dataFrame
print(standings_df.to_string(index=False))

 P                       Team  GP  W  D  L  F  A  GD  Pts
 1                 Arsenal FC  32 21  7  4 62 24  38   70
 2         Manchester City FC  31 19  7  5 63 28  35   64
 3       Manchester United FC  31 15 10  6 56 43  13   55
 4             Aston Villa FC  32 16  7  9 43 38   5   55
 5               Liverpool FC  32 15  7 10 52 42  10   52
 6                 Chelsea FC  32 13  9 10 53 41  12   48
 7               Brentford FC  32 13  8 11 48 44   4   47
 8                 Everton FC  32 13  8 11 39 37   2   47
 9  Brighton & Hove Albion FC  32 12 10 10 43 37   6   46
10             Sunderland AFC  32 12 10 10 33 36  -3   46
11            AFC Bournemouth  32 10 15  7 48 49  -1   45
12                  Fulham FC  32 13  5 14 43 46  -3   44
13          Crystal Palace FC  31 11  9 11 35 36  -1   42
14        Newcastle United FC  32 12  6 14 45 47  -2   42
15            Leeds United FC  31  7 12 12 37 48 -11   33
16       Nottingham Forest FC  32  8  9 15 32 44 -12   33
17         Wes

In [10]:
# Set up MySQL database connection
mysql_connection = pymysql.connect(
    database=MYSQL_DATABASE,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=int(MYSQL_PORT)
)

In [11]:
# Get a cursor from the database
cur = mysql_connection.cursor()

# Use SQL to create a table for the Premier League 
create_table_sql_query = """
    CREATE TABLE IF NOT EXISTS premier_league_standings_tbl (
            position            INT PRIMARY KEY,
            team                VARCHAR(255),
            games_played        INTEGER,
            wins                INTEGER,
            draws               INTEGER,
            losses              INTEGER,
            goals_for           INTEGER,
            goals_against       INTEGER,
            goal_difference     INTEGER,
            points              INTEGER
    );
"""

In [ ]:
# Run the SQL query 
cur.execute(create_table_sql_query)


# Commit the transaction 
mysql_connection.commit()

In [14]:
# Use SQL to insert data into the Premier League table 
insert_data_sql_query = """
    INSERT INTO premier_league_standings_tbl (
            position, team, games_played, wins, draws, losses, goals_for, goals_against, goal_difference, points              
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)

    ON DUPLICATE KEY UPDATE
    team               = VALUES(team), 
    games_played       = VALUES(games_played), 
    wins               = VALUES(wins), 
    draws              = VALUES(draws), 
    losses             = VALUES(losses), 
    goals_for          = VALUES(goals_for), 
    goals_against      = VALUES(goals_against), 
    goal_difference    = VALUES(goal_difference), 
    points             = VALUES(points)
"""

In [15]:
for idx, row in standings_df.iterrows():
    cur.execute(insert_data_sql_query, 
                (row['P'], 
                 row['Team'], 
                 row['GP'], 
                 row['W'], 
                 row['D'], 
                 row['L'], 
                 row['F'], 
                 row['A'], 
                 row['GD'], 
                 row['Pts'])
    )



# Commit the transaction 
mysql_connection.commit()

In [16]:
# Use SQL to create a view that updates the table rankings
drop_view_sql = "DROP VIEW IF EXISTS premier_league_standings_vw;"
create_ranked_standings_view_sql_query = """
    CREATE VIEW premier_league_standings_vw AS 
        SELECT 
            @rownum := @rownum + 1 AS position,
            team,
            games_played,
            wins,
            draws,
            losses,
            goals_for,
            goals_against,
            goal_difference,
            points
        FROM 
            premier_league_standings_tbl,
            (SELECT @rownum := 0) r
        ORDER BY points DESC, goal_difference DESC, goals_for DESC;
"""

In [17]:
# Run the SQL query 
cur.execute(drop_view_sql)
cur.execute(create_ranked_standings_view_sql_query)


# Commit the transaction 
mysql_connection.commit()

OperationalError: (1351, "View's SELECT contains a variable or parameter")